# KLASIFIKASI GENRE GAME BERDASARKAN SINOPSIS
---
**Author:** *Muhammad Wira Widhana [24.55.2717]*

> Alur penelitian mengikuti paper referensi (KNN):
> Pengumpulan Data -> Preprocessing -> Pembobotan TF-IDF -> Klasifikasi KNN -> Evaluasi (Confusion Matrix).
> Naive Bayes & SVM ditambahkan sebagai pembanding (sesuai bagian Saran paper).

## Library & Load Dataset

library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import ast
import math
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
)

from IPython.display import display

In [ ]:
# pertama kali saja:

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# digunakan di preprocessing
english_stopwords = set(stopwords.words('english'))
stemmer = PorterStemmer()

## Load dan Memeriksa Data (Data Loading & EDA)

Load Dataset

In [ ]:
# Load Dataset
try:
    df_raw = pd.read_csv('games_fixed.csv')
except Exception:
    # cadangan: lewati baris yang formatnya rusak pada ekspor CSV mentah
    df_raw = pd.read_csv('games_fixed.csv', engine='python', on_bad_lines='skip')

print("Ukuran awal dataset:", df_raw.shape)
df_raw.head()

>Penjelasan singkat: Membaca dataset `games_fixed.csv` yang merupakan dataset dari kaggle

Seleksi kolom yang dibutuhkan (judul, sinopsis, genre)

In [ ]:
# Seleksi kolom: judul, sinopsis, genre
df = df_raw[['Name', 'About the game', 'Genres']].copy()

df.rename(columns={
    'Name': 'title',
    'About the game': 'synopsis',
    'Genres': 'genre_raw'
}, inplace=True)

print("Jumlah data setelah seleksi kolom:", df.shape)
df.sample(5)

Cek missing values

In [ ]:
# Cek missing values
df.isna().sum()

Hapus baris yang tidak punya sinopsis atau genre

In [ ]:
# Hapus baris yang tidak punya sinopsis atau genre
df = df.dropna(subset=['synopsis', 'genre_raw']).reset_index(drop=True)

# Tambahan: hapus genre yang kosong secara konten, seperti [], [''], dll.
df['genre_raw'] = df['genre_raw'].astype(str).str.strip()

# Filter keluar string yang merepresentasikan list kosong
df = df[~df['genre_raw'].isin(['[]', "['']", '[""]', '', 'nan'])].reset_index(drop=True)

print("Jumlah data setelah drop NA dan genre kosong:", df.shape)

## Normalisasi Genre

Mengikuti paper: 4 genre dan single-label (satu data = satu genre).
Di sini dipakai 4 genre gameplay murni: **action, adventure, rpg, simulation**.

In [ ]:
# Genre target: 4 genre gameplay murni (tanpa 'indie' & 'casual')
target_genres = {'action', 'adventure', 'rpg', 'simulation'}

def parse_genre_list(genre_str):
    '''
    Parse string genre seperti "['Action', 'Indie']" menjadi list python ['action', 'indie'].
    Parsing aman menggunakan ast.literal_eval (bukan replace manual).
    '''
    if pd.isna(genre_str):
        return []

    # Jika sudah list, kembalikan langsung
    if isinstance(genre_str, list):
        raw_list = genre_str
    else:
        text = str(genre_str).strip()
        try:
            # Coba parse sebagai literal Python
            raw_list = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            # Cadangan: pisah manual dengan koma
            raw_list = [g.strip() for g in text.split(',')]

    # Normalisasi ke huruf kecil
    return [g.strip().lower() for g in raw_list if isinstance(g, str)]

# Terapkan parsing dan filter ke genre target
df['genre_list'] = df['genre_raw'].apply(parse_genre_list)
df['genre_filtered'] = df['genre_list'].apply(lambda lst: [g for g in lst if g in target_genres])
df['genre_count'] = df['genre_filtered'].apply(len)

df[['genre_raw', 'genre_list', 'genre_filtered', 'genre_count']].head()

In [ ]:
# Untuk menjaga single-label classification sesuai paper,
# hanya ambil data yang memiliki tepat satu genre target
df_single = df[df['genre_count'] == 1].copy().reset_index(drop=True)

# Ambil genre utama
df_single['genre_main'] = df_single['genre_filtered'].str[0]

print("Jumlah data setelah filter subset genre dan single-label:", df_single.shape)
df_single[['title', 'synopsis', 'genre_raw', 'genre_filtered', 'genre_main']].head(10)

## Pembersihan Data

Buang sinopsis non-Inggris, placeholder, terlalu pendek, dan duplikat
agar data layak diklasifikasi (paper memakai sinopsis berbahasa Inggris).

In [ ]:
# Filter sederhana: sinopsis dominan huruf Latin & tidak terlalu pendek
def basic_english_filter(text):
    if not isinstance(text, str):
        return False
    clean = re.sub(r'[^A-Za-z\s]', ' ', text)
    clean = re.sub(r'\s+', ' ', clean).strip()
    if len(clean.split()) < 3:
        return False
    return (len(clean) / max(len(text), 1)) >= 0.7

df_single['is_english'] = df_single['synopsis'].apply(basic_english_filter)
df_single = df_single[df_single['is_english']].drop(columns=['is_english']).reset_index(drop=True)
print("Jumlah data setelah filter bahasa Inggris:", df_single.shape)

In [ ]:
# Buang sinopsis placeholder, terlalu pendek, dan duplikat
MIN_WORDS, MIN_CHARS = 10, 40
_syn = df_single['synopsis'].astype(str).str.strip()
word_count = _syn.str.split().apply(len)
char_count = _syn.str.len()

placeholder_re = re.compile(r'^(no description( available)?|coming soon|tba|tbd|n/?a|none|\.+|-+)$', re.I)
is_placeholder = _syn.apply(lambda s: bool(placeholder_re.match(s)))
too_short = (word_count < MIN_WORDS) | (char_count < MIN_CHARS)
is_dup = _syn.str.lower().duplicated(keep='first')

remove_mask = is_placeholder | too_short | is_dup
before = len(df_single)
df_single = df_single[~remove_mask].reset_index(drop=True)
print(f"Data sebelum: {before} -> sesudah: {len(df_single)} (dibuang {before - len(df_single)})")

sebaran genre

In [ ]:
df_single['genre_main'].value_counts()

In [ ]:
# Visualisasi distribusi genre (single-label, 4 genre target)
genre_counts = df_single['genre_main'].value_counts()

plt.figure(figsize=(8, 5))
ax = genre_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribusi Genre Game (4 Genre Target)')
plt.xlabel('Genre')
plt.ylabel('Jumlah Game')
plt.xticks(rotation=45)

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.tight_layout()
plt.show()

## Preprocessing

Urutan tahapan mengikuti paper:
`Case Folding -> Cleansing -> Stopword Removal -> Stemming -> Tokenization`

Disediakan 3 skenario untuk menguji kombinasi fitur preprocessing:
- **S1**: Case folding + Cleansing + Stopword removal + Tokenization
- **S2**: Case folding + Cleansing + Stemming + Tokenization
- **S3 (sesuai paper)**: Case folding + Cleansing + Stopword removal + Stemming + Tokenization

In [ ]:
# 1) Case Folding: ubah semua huruf menjadi huruf kecil
def case_folding(text):
    return str(text).lower()

# 2) Cleansing: hapus URL, angka, tanda baca, dan karakter non-huruf
def cleansing(text):
    text = re.sub(r'http\S+', ' ', text)      # hapus URL
    text = re.sub(r'[^a-z\s]', ' ', text)      # hapus angka & tanda baca
    text = re.sub(r'\s+', ' ', text).strip()   # rapikan spasi berlebih
    return text

# 3) Stopword Removal: buang kata umum yang tidak informatif
def stopword_removal(text):
    return ' '.join(w for w in text.split() if w not in english_stopwords)

# 4) Stemming: kembalikan kata ke bentuk dasar (Porter Stemmer)
def stemming(text):
    return ' '.join(stemmer.stem(w) for w in text.split())

# 5) Tokenization: pisahkan teks menjadi token kata
def tokenization(text):
    return word_tokenize(text)

In [ ]:
def preprocess(text, use_stopword=True, use_stemming=True):
    '''Pipeline preprocessing sesuai urutan paper.'''
    text = case_folding(text)            # 1. Case Folding
    text = cleansing(text)               # 2. Cleansing
    if use_stopword:
        text = stopword_removal(text)    # 3. Stopword Removal
    if use_stemming:
        text = stemming(text)            # 4. Stemming
    tokens = tokenization(text)          # 5. Tokenization
    return ' '.join(tokens)

In [ ]:
# Terapkan ketiga skenario preprocessing
df_single['synopsis_S1'] = df_single['synopsis'].apply(lambda x: preprocess(x, True,  False))  # stopword only
df_single['synopsis_S2'] = df_single['synopsis'].apply(lambda x: preprocess(x, False, True))   # stemming only
df_single['synopsis_S3'] = df_single['synopsis'].apply(lambda x: preprocess(x, True,  True))   # SESUAI PAPER

df_single[['genre_main', 'synopsis', 'synopsis_S3']].head()

>Penjelasan singkat: Skenario S3 (stopword removal + stemming) adalah skenario utama sesuai paper.

## Pembobotan TF-IDF

Skenario utama = S3 (sesuai paper). Data dibagi 80% latih : 20% uji (stratified),
lalu TF-IDF di-`fit` hanya pada data latih.

In [ ]:
# Skenario utama = S3 (sesuai paper)
TEXT_COL = 'synopsis_S3'
X_text = df_single[TEXT_COL]
y = df_single['genre_main']

# Split 80:20 stratified
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data latih:", X_train_text.shape[0])
print("Jumlah data uji  :", X_test_text.shape[0])

In [ ]:
# Pembobotan TF-IDF (fit pada train, transform pada test)
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.95
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("Dimensi matriks TF-IDF latih:", X_train.shape)
print("Dimensi matriks TF-IDF uji  :", X_test.shape)

## Klasifikasi

Model utama **KNN** mengikuti paper (metric **euclidean**, **k = akar jumlah data latih**).
Naive Bayes & SVM ditambahkan sebagai pembanding.

In [ ]:
# Nilai k mengikuti aturan paper: akar dari jumlah data latih
k = int(round(math.sqrt(X_train.shape[0])))
print("Nilai k (akar jumlah data latih):", k)

In [ ]:
# Model utama: K-Nearest Neighbors (sesuai paper)
knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

In [ ]:
# Pembanding 1: Naive Bayes
nb_clf = MultinomialNB(alpha=0.1)
nb_clf.fit(X_train, y_train)
y_pred_nb = nb_clf.predict(X_test)

In [ ]:
# Pembanding 2: Support Vector Machine
svm_clf = LinearSVC(class_weight='balanced', max_iter=5000)
svm_clf.fit(X_train, y_train)
y_pred_svm = svm_clf.predict(X_test)

## Evaluasi (Confusion Matrix)

Evaluasi memakai: (a) Confusion Matrix, (b) **perhitungan manual TP/FP/FN/TN per kelas**
seperti tabel di paper, dan (c) ringkasan Accuracy, Precision, Recall, F1-Score.

In [ ]:
def manual_confusion_matrix(y_true, y_pred, labels):
    '''
    Hitung TP, FP, FN, TN per kelas secara MANUAL dari confusion matrix,
    persis seperti contoh tabel pada paper referensi.
    - TP: prediksi benar untuk kelas tsb (diagonal)
    - FP: data kelas lain yang diprediksi sebagai kelas tsb (kolom - TP)
    - FN: data kelas tsb yang diprediksi sebagai kelas lain (baris - TP)
    - TN: sisanya
    '''
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    total = cm.sum()
    rows = []
    for i, label in enumerate(labels):
        TP = int(cm[i, i])
        FP = int(cm[:, i].sum() - TP)
        FN = int(cm[i, :].sum() - TP)
        TN = int(total - TP - FP - FN)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        rows.append({
            'Kelas': label,
            'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
            'Precision': round(precision, 4),
            'Recall': round(recall, 4),
            'F1-Score': round(f1, 4),
        })
    return pd.DataFrame(rows)

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    '''Tampilkan confusion matrix, tabel manual TP/FP/FN/TN, dan ringkasan metrik.'''
    labels = sorted(pd.Series(y_true).unique())
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    print('=' * 60)
    print('MODEL:', model_name)
    print('=' * 60)

    # (a) Confusion matrix (heatmap)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

    # (b) Tabel manual TP/FP/FN/TN per kelas (seperti paper)
    print('Perhitungan Manual Confusion Matrix (per kelas):')
    tabel = manual_confusion_matrix(y_true, y_pred, labels)
    display(tabel)

    # (c) Ringkasan metrik keseluruhan (macro average)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"Accuracy : {acc * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall   : {rec * 100:.2f}%")
    print(f"F1-Score : {f1 * 100:.2f}%")

### Evaluasi KNN (model utama paper)

In [ ]:
evaluate_model(y_test, y_pred_knn, 'K-Nearest Neighbors')

### Evaluasi Naive Bayes (pembanding)

In [ ]:
evaluate_model(y_test, y_pred_nb, 'Naive Bayes')

### Evaluasi SVM (pembanding)

In [ ]:
evaluate_model(y_test, y_pred_svm, 'Support Vector Machine')

>Penjelasan singkat: Tabel manual menampilkan TP/FP/FN/TN tiap genre, lalu diringkas menjadi Accuracy, Precision, Recall, dan F1-Score.

## Analisis Tambahan

Bagian ini di luar paper, namun memperkuat analisis: validasi silang K-Fold,
MSE & RMSE, dan perbandingan antar skenario preprocessing.

### Validasi Silang K-Fold (Stratified, k=5)

In [ ]:
X_cv = df_single['synopsis_S3']
y_cv = df_single['genre_main']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'Accuracy': 'accuracy',
    'Precision': 'precision_macro',
    'Recall': 'recall_macro',
    'F1-Score': 'f1_macro',
}

k_cv = int(round(math.sqrt(len(X_cv) * 0.8)))
cv_models = {
    'KNN':         KNeighborsClassifier(n_neighbors=k_cv, metric='euclidean'),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM':  LinearSVC(class_weight='balanced', max_iter=5000),
}

cv_rows = []
for name, model in cv_models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True, min_df=3, max_df=0.95)),
        ('clf', model),
    ])
    scores = cross_validate(pipe, X_cv, y_cv, cv=skf, scoring=scoring)
    row = {'Model': name}
    for label in scoring:
        row[label] = f"{scores[f'test_{label}'].mean() * 100:.2f}% +/- {scores[f'test_{label}'].std() * 100:.2f}%"
    cv_rows.append(row)

pd.DataFrame(cv_rows)

### Tabel MSE & RMSE

In [ ]:
le = LabelEncoder().fit(df_single['genre_main'])

def hitung_mse_rmse(y_true, y_pred):
    mse = mean_squared_error(le.transform(y_true), le.transform(y_pred))
    return mse, np.sqrt(mse)

mse_rows = []
for model_name, y_pred in [('KNN', y_pred_knn), ('Naive Bayes', y_pred_nb), ('Linear SVM', y_pred_svm)]:
    mse, rmse = hitung_mse_rmse(y_test, y_pred)
    mse_rows.append({'Model': model_name, 'MSE': round(mse, 4), 'RMSE': round(rmse, 4)})

pd.DataFrame(mse_rows).sort_values('MSE').reset_index(drop=True)

### Perbandingan Antar Skenario Preprocessing (S1, S2, S3)

In [ ]:
scenarios = {
    'S1_stopword_only': 'synopsis_S1',
    'S2_stemming_only': 'synopsis_S2',
    'S3_stopword_stem': 'synopsis_S3',
}
le_cmp = LabelEncoder().fit(df_single['genre_main'])
results = []

for scen_name, text_col in scenarios.items():
    X_text_s = df_single[text_col]
    y_s = df_single['genre_main']
    Xtr_t, Xte_t, ytr, yte = train_test_split(X_text_s, y_s, test_size=0.2, random_state=42, stratify=y_s)
    tfidf_s = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True, min_df=3, max_df=0.95)
    Xtr = tfidf_s.fit_transform(Xtr_t)
    Xte = tfidf_s.transform(Xte_t)
    k_s = int(round(math.sqrt(Xtr.shape[0])))

    models = {
        'KNN':         KNeighborsClassifier(n_neighbors=k_s, metric='euclidean'),
        'Naive Bayes': MultinomialNB(alpha=0.1),
        'Linear SVM':  LinearSVC(class_weight='balanced', max_iter=5000),
    }
    for model_name, model in models.items():
        model.fit(Xtr, ytr)
        yp = model.predict(Xte)
        mse = mean_squared_error(le_cmp.transform(yte), le_cmp.transform(yp))
        results.append({
            'Scenario': scen_name,
            'Model': model_name,
            'Accuracy': accuracy_score(yte, yp),
            'Precision': precision_score(yte, yp, average='macro', zero_division=0),
            'Recall': recall_score(yte, yp, average='macro', zero_division=0),
            'F1': f1_score(yte, yp, average='macro', zero_division=0),
            'MSE': mse,
            'RMSE': np.sqrt(mse),
        })

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_display = results_df.copy()
for c in ['Accuracy', 'Precision', 'Recall', 'F1']:
    results_display[c] = results_display[c].apply(lambda x: f"{x * 100:.2f}%")
for c in ['MSE', 'RMSE']:
    results_display[c] = results_display[c].apply(lambda x: f"{x:.4f}")
results_display

### Implementasi (Inferensi Genre)

In [ ]:
# Otomatis memakai model dengan F1 tertinggi dari tabel perbandingan
best = results_df.sort_values('F1', ascending=False).iloc[0]
_prep = {
    'S1_stopword_only': (True, False),
    'S2_stemming_only': (False, True),
    'S3_stopword_stem': (True, True),
}[best['Scenario']]
best_k = int(round(math.sqrt(len(df_single) * 0.8)))

final_tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True, min_df=3, max_df=0.95)
final_model = {
    'KNN':         KNeighborsClassifier(n_neighbors=best_k, metric='euclidean'),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM':  LinearSVC(class_weight='balanced', max_iter=5000),
}[best['Model']]
final_model.fit(final_tfidf.fit_transform(df_single[scenarios[best['Scenario']]]), df_single['genre_main'])

def predict_genre(text):
    vec = final_tfidf.transform([preprocess(text, *_prep)])
    return final_model.predict(vec)[0]

contoh = "An open-world fantasy adventure where you explore dungeons, level up your hero, and battle monsters."
print(f"Model terbaik  : {best['Model']} ({best['Scenario']}, F1={best['F1'] * 100:.2f}%)")
print(f"Sinopsis       : {contoh}")
print(f"Prediksi genre : {predict_genre(contoh)}")